# 🦅 PREDATOR Analytics v63.1-ELITE: Full-Stack Kaggle Node (CPU)
**Status**: Hardened Deployment Mode
**Architecture**: K3s + Helm + 8 DBs + zrok Tunnel

In [ ]:
# 1. Встановлення системних компонентів
!curl -sfL https://get.k3s.io | INSTALL_K3S_SKIP_ENABLE=true sh -
!curl https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash
!wget -q https://github.com/openziti/zrok/releases/download/v0.4.42/zrok_0.4.42_linux_amd64.tar.gz && tar -xzf zrok_*.tar.gz && chmod +x zrok

import os, subprocess, time, threading

WORKING_DIR = "/kaggle/working"
DATA_DIR = f"{WORKING_DIR}/k3s-data"
KUBECONFIG = f"{DATA_DIR}/kubeconfig.yaml"
os.environ['KUBECONFIG'] = KUBECONFIG
os.makedirs(DATA_DIR, exist_ok=True)

print("🛸 Запуск K3s Server...")
k3s_log = open(f"{WORKING_DIR}/k3s.log", "w")
k3s_cmd = ["k3s", "server", "--write-kubeconfig-mode", "644", "--data-dir", DATA_DIR, "--snapshotter", "native", "--disable", "traefik"]
subprocess.Popen(k3s_cmd, stdout=k3s_log, stderr=k3s_log)

for i in range(60):
    if os.path.exists(KUBECONFIG):
        if subprocess.run(["kubectl", "get", "nodes"], capture_output=True).returncode == 0:
            print("✅ K3s API Ready!")
            break
    time.sleep(3)

# 2. Підготовка чартів
print("🐙 Клонування та збірка залежностей...")
!rm -rf repo && git clone https://github.com/dima1203oleg/predator-analytics.git repo
os.chdir(f"{WORKING_DIR}/repo/deploy/helm/predator")
!/usr/local/bin/helm repo add bitnami https://charts.bitnami.com/bitnami
!/usr/local/bin/helm repo add redpanda https://charts.redpanda.com
!/usr/local/bin/helm repo update
!/usr/local/bin/helm dependency build

# 3. Деплой
values_content = "global:\n  domain: kaggle.predator.ua\npostgresql: {enabled: true}\nneo4j: {enabled: true}\ncoreApi: {replicaCount: 1}\nclickhouse: {enabled: true}\nopensearch: {enabled: true, singleNode: true}\nqdrant: {enabled: true}\nredis: {enabled: true}\nminio: {enabled: true}\nredpanda: {enabled: true}"
with open('values-kaggle.yaml', 'w') as f: f.write(values_content)

print("📦 Інсталяція PREDATOR Elite...")
!/usr/local/bin/helm install predator . -f values.yaml -f values-kaggle.yaml --namespace predator-elite --create-namespace

# 4. Тунель та моніторинг (НЕ ПЕРЕРИВАЄ РОБОТУ)
print("🌐 Запуск тунелю zrok та моніторингу статусу...")
!/kaggle/working/zrok enable 1eeje4um7yvA

def monitor_and_tunnel():
    # Чекаємо сервіс
    while subprocess.run(["kubectl", "get", "svc", "-n", "predator-elite", "predator-core-api"], capture_output=True).returncode != 0:
        time.sleep(5)
    
    subprocess.Popen(["kubectl", "port-forward", "-n", "predator-elite", "svc/predator-core-api", "8000:8000"])
    time.sleep(5)
    
    p = subprocess.Popen(["/kaggle/working/zrok", "share", "public", "http://localhost:8000", "--headless"], stdout=subprocess.PIPE, text=True)
    for line in p.stdout:
        if "access your share at" in line:
            url = line.split('access your share at')[-1].strip()
            print(f"\n🚀 PREDATOR LIVE URL: {url}")
            break

threading.Thread(target=monitor_and_tunnel, daemon=True).start()

# ЦИКЛ ОЧІКУВАННЯ (KEEP ALIVE)
try:
    while True:
        print("\n--- [Поточний статус подів] ---")
        !kubectl get pods -n predator-elite
        time.sleep(30)
except KeyboardInterrupt:
    print("🛑 Зупинка за запитом користувача.")